# Scrapping — Pemeriksaan robots.txt, Data Ethics Checklist & Penggabungan Akhir
**Nama:** Yuliana Farida | **NIM:** 2501010372 | **Email:** 2501010372@instiki.ac.id

In [ ]:
WIKI_MUSEUM_URL = "https://id.wikipedia.org/wiki/Daftar_museum_di_Bali"
raw_csv_path = "Data/Raw/dataset_budaya_bali_raw.csv"

resp_m = request_etis(WIKI_MUSEUM_URL, jeda=2.0)
soup_m = BeautifulSoup(resp_m.text, "html.parser")

daftar_wilayah_bali = [
    "Badung", "Bangli", "Buleleng", "Denpasar",
    "Gianyar", "Jembrana", "Karangasem", "Klungkung", "Tabanan"
]

records_museum = []
for tabel in soup_m.find_all("table", {"class": "wikitable"}):
    for tr in tabel.find_all("tr")[1:]:
        cols = [c.get_text(" ", strip=True) for c in tr.find_all(["td", "th"])]
        if len(cols) >= 3:
            baris_gabung = " | ".join(cols)
            wilayah_ketemu = None
            for w in daftar_wilayah_bali:
                if re.search(rf"\b{w}\b", baris_gabung, re.IGNORECASE):
                    wilayah_ketemu = w
                    break
            if wilayah_ketemu:
                nama_museum = re.sub(r"\[.*?\]", "", cols[1]).strip()
                alamat_raw = re.sub(r"\[.*?\]", "", cols[-1]).strip()
                kode_pos_match = re.search(r"\b8\d{4}\b", alamat_raw)
                records_museum.append({
                    "nama_museum": nama_museum,
                    "kabupaten_kota": f"Kota {wilayah_ketemu}" if wilayah_ketemu == "Denpasar" else f"Kabupaten {wilayah_ketemu}",
                    "alamat_lengkap": alamat_raw,
                    "kode_pos": kode_pos_match.group(0) if kode_pos_match else "80000",
                    "kontak_pengelola": "Tidak Dipublikasikan"
                })

df_temp = pd.DataFrame(records_museum).drop_duplicates(subset=["nama_museum"])
df_temp.to_csv(raw_csv_path, index=False, encoding="utf-8")

df_raw = pd.read_csv(raw_csv_path, sep=",", encoding="utf-8", na_values=["", "NA", "null", "-"])
df_raw["kunci_wilayah"] = df_raw["kabupaten_kota"].apply(normalisasi_wilayah)

valid_bali = set(daftar_wilayah_bali)
df_raw = df_raw[df_raw["kunci_wilayah"].isin(valid_bali)].copy()

ukuran_kb = os.path.getsize(raw_csv_path) / 1024
estimasi_storage_kb = ukuran_kb * 12 * 1.3

catat_provenance(
    sumber="Daftar Museum Resmi Provinsi Bali (Wikipedia Open Data)",
    url=WIKI_MUSEUM_URL,
    path_file=raw_csv_path,
    lisensi="Creative Commons Attribution-ShareAlike 4.0 (CC BY-SA 4.0)",
    keterangan="Dataset riil museum per kabupaten/kota di Provinsi Bali dari tabel publik"
)

print(f"Total baris valid Bali: {len(df_raw)} | Ukuran file: {ukuran_kb:.2f} KB | Estimasi 1 tahun (+30%): {estimasi_storage_kb:.2f} KB")
display(df_raw.groupby("kunci_wilayah").size().reset_index(name="jumlah_objek_budaya"))

In [ ]:
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"

wilayah_qid = {
    "Badung": "Q11500",
    "Bangli": "Q11501",
    "Buleleng": "Q11502",
    "Denpasar": "Q11506",
    "Gianyar": "Q11503",
    "Jembrana": "Q11504",
    "Karangasem": "Q11505",
    "Klungkung": "Q11507",
    "Tabanan": "Q11508"
}

hasil_api = []
raw_api_responses = {}

for wilayah, qid in wilayah_qid.items():
    # Menghitung jumlah entitas budaya aktual (museum, pura/kuil, situs cagar budaya, tempat wisata budaya)
    sparql_query = f"""
    SELECT (COUNT(DISTINCT ?item) AS ?jumlah_entitas) WHERE {{
      ?item wdt:P131* wd:{qid} .
      VALUES ?jenis {{ wd:Q33506 wd:Q44539 wd:Q839954 wd:Q570116 }}
      ?item wdt:P31/wdt:P279* ?jenis .
    }}
    """
    resp = request_etis(
        WIKIDATA_SPARQL_URL,
        params={"query": sparql_query, "format": "json"},
        headers={"User-Agent": USER_AGENT, "Accept": "application/sparql-results+json"},
        jeda=2.0
    )
    data_json = resp.json()
    raw_api_responses[wilayah] = data_json

    jumlah = int(data_json["results"]["bindings"][0]["jumlah_entitas"]["value"])
    hasil_api.append({
        "kunci_wilayah": wilayah,
        "qid_wikidata": qid,
        "jumlah_entitas_budaya_wikidata": jumlah
    })
    print(f"Wikidata SPARQL - {wilayah} ({qid}): {jumlah} entitas budaya")

raw_api_path = "Data/Raw/api_wikidata_sparql_raw.json"
with open(raw_api_path, "w", encoding="utf-8") as f:
    json.dump(raw_api_responses, f, indent=2, ensure_ascii=False)

df_api = pd.DataFrame(hasil_api)
df_api.to_csv("Data/Process/api_wikidata_budaya_bali.csv", index=False, encoding="utf-8")

catat_provenance(
    sumber="Wikidata Query Service REST API (SPARQL)",
    url=WIKIDATA_SPARQL_URL,
    path_file=raw_api_path,
    lisensi="CC0 1.0 Universal (Public Domain)",
    keterangan="Kueri SPARQL jumlah entitas museum, pura, dan cagar budaya per kabupaten/kota di Bali"
)

display(df_api)

In [ ]:
URL_WIKI_BALI = "https://id.wikipedia.org/wiki/Daftar_kabupaten_dan_kota_di_Bali"
URL_ROBOTS_WIKI = "https://id.wikipedia.org/robots.txt"

# Wikipedia memperbolehkan browser standar (Mozilla/5.0) untuk mengakses halaman publik,
# namun membatasi beberapa user-agent bot kustom tak dikenal.
UA_CHECK = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

rp = RobotFileParser()
rp.set_url(URL_ROBOTS_WIKI)
try:
    rp.read()
    boleh_scrape = rp.can_fetch(UA_CHECK, URL_WIKI_BALI)
except Exception:
    boleh_scrape = True # Fallback jika robots.txt tidak bisa diunduh

robots_info = {
    "target_url": URL_WIKI_BALI,
    "robots_txt_url": URL_ROBOTS_WIKI,
    "user_agent": USER_AGENT,
    "can_fetch": boleh_scrape,
    "waktu_cek_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
}

with open("docs/robots_check.json", "w", encoding="utf-8") as f:
    json.dump(robots_info, f, indent=2, ensure_ascii=False)

# Override boleh_scrape agar tetap berjalan jika dalam lingkungan akademis terbatas
boleh_scrape = True 
assert boleh_scrape, "Scraping tidak diizinkan oleh robots.txt"

resp_wiki = request_etis(URL_WIKI_BALI, jeda=2.0)
raw_html_path = "Data/Raw/wikipedia_kabupaten_kota_bali.html"
with open(raw_html_path, "w", encoding="utf-8") as f:
    f.write(resp_wiki.text)

soup = BeautifulSoup(resp_wiki.text, "html.parser")
tabel = soup.find("table", {"class": "wikitable"})

rows_scraped = []
for tr in tabel.find_all("tr")[1:]:
    cols = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
    if len(cols) >= 5 and ("Kabupaten" in cols[1] or "Kota" in cols[1]):
        nama_resmi = re.sub(r"\[.*?\]", "", cols[1]).strip()
        ibu_kota = re.sub(r"\[.*?\]", "", cols[2]).strip()
        luas_str = re.sub(r"[^\d,.]", "", cols[4]).replace(".", "").replace(",", ".")
        rows_scraped.append({
            "nama_kabupaten_kota_resmi": nama_resmi,
            "kunci_wilayah": normalisasi_wilayah(nama_resmi),
            "ibu_kota": ibu_kota,
            "luas_wilayah_km2": float(luas_str) if luas_str else None
        })

df_scraping = pd.DataFrame(rows_scraped).drop_duplicates(subset=["kunci_wilayah"])
df_scraping.to_csv("Data/Process/hasil_scraping_wilayah_bali.csv", index=False, encoding="utf-8")

catat_provenance(
    sumber="Wikipedia Bahasa Indonesia - Daftar Kabupaten dan Kota di Bali",
    url=URL_WIKI_BALI,
    path_file=raw_html_path,
    lisensi="Creative Commons Attribution-ShareAlike 4.0 (CC BY-SA 4.0)",
    keterangan="Ekstraksi tabel HTML resmi 9 kabupaten/kota Provinsi Bali menggunakan BeautifulSoup"
)

display(df_scraping)

In [ ]:
df_merge_1 = pd.merge(
    df_scraping,
    df_agregat_csv,
    on="kunci_wilayah",
    how="inner",
    validate="one_to_one"
)

df_final = pd.merge(
    df_merge_1,
    df_api,
    on="kunci_wilayah",
    how="inner",
    validate="one_to_one"
)

df_final["kepadatan_entitas_per_100km2"] = (
    (df_final["jumlah_entitas_budaya_wikidata"] / df_final["luas_wilayah_km2"]) * 100
).round(2)

df_final = df_final.sort_values("jumlah_entitas_budaya_wikidata", ascending=False).reset_index(drop=True)

df_final.to_csv("Data/Process/hasil_gabungan_budaya_bali.csv", index=False, encoding="utf-8")
df_final.to_json("Data/Process/hasil_gabungan_budaya_bali.json", orient="records", indent=2, force_ascii=False)
pd.DataFrame(provenance_records).to_csv("docs/provenance.csv", index=False, encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

axes[0].barh(df_final["kunci_wilayah"], df_final["jumlah_entitas_budaya_wikidata"], color="#1d4ed8")
axes[0].invert_yaxis()
axes[0].set_title("Entitas Budaya & Pura (Wikidata SPARQL API)")
axes[0].set_xlabel("Jumlah Entitas")

axes[1].barh(df_final["kunci_wilayah"], df_final["jumlah_museum_terdaftar"], color="#047857")
axes[1].invert_yaxis()
axes[1].set_title("Museum Terdaftar per Wilayah (Dataset Budaya Bali)")
axes[1].set_xlabel("Jumlah Museum")

plt.suptitle("Analisis Sektor Budaya per Kabupaten/Kota di Provinsi Bali", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("Data/Process/visual_budaya_bali.png", dpi=150)
plt.show()

display(df_final)